In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import Tensor
from jaxtyping import Float, Int

In [ ]:

class RNNModel(nn.Module):
    def __init__(self, vocab_size: int = 65, n_embed: int = 32, hidden_size: int = 64) -> None:
      super().__init__()    
      self.hidden_size = hidden_size

      self.table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embed) # (65, 32)
      self.linear1 = nn.Linear(n_embed + self.hidden_size, hidden_size) # (96{32 + 64}, 64) -> (bs, 64)
      self.tan_h = nn.Tanh() 
      # Create linear2: projects from hidden space to vocab_size to produce logits (one score per token)
      self.linear2 = nn.Linear(hidden_size, vocab_size) # (64, 65)

     # idx is tensor (B, T), batch and timestampt/sequence
      def forward(idx: Tensor, targets=None):
        B, T = idx.shape
        h0 = torch.zeros([B, self.hidden_size], device=idx.device) # (B, hidden_size) hidden memory, match to idx device to prevent mismatch
        
        # Embed all tokens at once using the embedding table
        embed = self.table(idx) # (B,T,n_embed) each id becomes a vector
        # Create an empty list to collect the logit output at each timestep
        logits = []
        # Loop over each timestep t from 0 to T-1
        for t in range(T): 
            pass
            # embedding for timestep t
            # B token embeddings, one per sequence at position t
            x_t = embed[:, t, :]  # B,n_embed

            # Concatenate x_t and the previous hidden state h along the last dimension
            # x_t is shape (B, n_embed), h is shape (B, hidden_size)
            combined = torch.cat([x_t, h0], dim=-1)
            # After cat: (B, n_embed + hidden_size)
            # This is the core RNN idea: the input and memory are merged into one vector
            # so the network can use both "what am I seeing now" and "what have I seen before"

            # Pass the concatenated vector through linear1, then apply Tanh activation
            # linear1 maps (B, n_embed + hidden_size) → (B, hidden_size)
            # Tanh squashes values to (-1, 1), keeping the hidden state bounded across many timesteps
            # The result becomes the new h — this is the updated memory of the RNN

            # Pass the new hidden state through linear2 to get logits for this timestep
            # linear2 maps (B, hidden_size) → (B, vocab_size)
            # Each value is a raw score for how likely each token is to come next

            # Append the logit tensor for this timestep to the logits list

        # Stack the list of T logit tensors into a single tensor along dimension 1
        # Goes from a list of T tensors of shape (B, vocab_size) → (B, T, vocab_size)
        # Now we have one prediction per timestep, for every item in the batch

        # Initialize loss as None (only computed if targets are provided)

        # If targets were passed in, compute the cross entropy loss
            # Unpack logits shape into B, T, C (C = vocab_size)
            # Reshape logits from (B, T, C) → (B*T, C) — cross_entropy expects (N, C)
            # Reshape targets from (B, T) → (B*T,) — cross_entropy expects (N,) for class indices
            # Call F.cross_entropy on the reshaped logits and targets

        # Return logits and loss


    # Define count_params: sums the total number of learnable parameters in the model
    # p.numel() gives the number of elements in each parameter tensor


In [3]:
emb = nn.Embedding(12, 6)

In [4]:
emb.weight

Parameter containing:
tensor([[-1.9490, -0.1500,  1.6735,  1.7063, -1.1451, -1.9368],
        [ 0.4057, -0.3269, -1.1988,  0.8812, -0.2675, -0.2132],
        [ 0.1814, -1.5926, -1.3695,  0.6432, -0.4539, -1.2895],
        [ 0.7306,  0.5385,  0.5760,  0.8859,  0.1921,  0.1919],
        [ 0.5913,  0.0403,  1.6371,  0.3330, -0.4794, -1.6811],
        [ 0.0152, -0.6284,  0.4413,  1.1056, -0.7835,  1.2850],
        [-1.7192,  0.3629, -1.5519, -0.1967, -0.5641,  1.6842],
        [ 0.7193, -1.3841, -0.9282,  0.6495, -0.3193, -0.9458],
        [-0.9212, -0.0278, -0.0537, -0.0651,  0.0722, -0.8689],
        [-1.0346,  1.4322,  0.1052, -1.2517,  0.9330, -1.4159],
        [ 1.3112, -0.3239,  0.0183,  1.7061,  1.4064, -0.0858],
        [-1.4038,  0.7561, -0.5565,  0.0725,  2.2286,  2.1837]],
       requires_grad=True)

In [5]:
emb(torch.tensor([[1,2,3], [1,2,3]]))

tensor([[[ 0.4057, -0.3269, -1.1988,  0.8812, -0.2675, -0.2132],
         [ 0.1814, -1.5926, -1.3695,  0.6432, -0.4539, -1.2895],
         [ 0.7306,  0.5385,  0.5760,  0.8859,  0.1921,  0.1919]],

        [[ 0.4057, -0.3269, -1.1988,  0.8812, -0.2675, -0.2132],
         [ 0.1814, -1.5926, -1.3695,  0.6432, -0.4539, -1.2895],
         [ 0.7306,  0.5385,  0.5760,  0.8859,  0.1921,  0.1919]]],
       grad_fn=<EmbeddingBackward0>)

In [6]:
vocab_size: int = 65
n_embed: int = 32
hidden_size: int = 64

In [7]:
batch = torch.randint(0, 5, (4,8))

In [8]:
batch

tensor([[1, 1, 4, 2, 2, 1, 1, 3],
        [3, 2, 4, 0, 0, 4, 0, 3],
        [0, 1, 0, 3, 1, 1, 3, 1],
        [0, 1, 0, 3, 3, 3, 2, 2]])

In [9]:
B, T = batch.shape
B, T

(4, 8)

In [10]:
table = nn.Embedding(num_embeddings=65, embedding_dim=32) # (65, 32)

In [11]:
embed = table(batch)
embed.shape

torch.Size([4, 8, 32])

In [12]:
# batch at position 0 
x_t = embed[:, 0, :]

In [13]:
# B(4) token embeddings at position 0
x_t.shape

torch.Size([4, 32])

In [20]:
h0 = torch.zeros([4, 64]) # B, hidden_size
h0.shape

torch.Size([4, 64])

In [21]:
x_t.shape, h0.shape

(torch.Size([4, 32]), torch.Size([4, 64]))

In [24]:
combined = torch.cat([x_t, h0], dim=-1)
combined.shape

torch.Size([4, 96])

In [25]:
combined

tensor([[-1.5752, -0.4979, -2.3956, -0.1368,  0.9251,  1.2088, -0.5386, -1.2101,
         -0.1747,  0.0565, -0.1878, -0.6992, -0.0662, -0.2662, -0.8034,  2.3788,
          0.1658,  1.2917, -0.1721, -0.9621, -2.1527, -1.1691,  0.5768,  0.3627,
          0.6397,  1.0240, -0.7732, -0.0318,  0.4957,  1.9355,  0.0153, -0.1329,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0474, -1.1366, 

In [ ]:
linear1 = nn.Linear(32 + 64, 64) # (96{32 + 64}, 64) -> ()

In [ ]:
lin1_out = linear1(lin_tens)
lin1_out.max()

In [ ]:
tan_h = nn.Tanh()
tanlin = tan_h(lin1_out)
tanlin.shape
tanlin.max()

In [ ]:
lin2 = nn.Linear(hidden_size, vocab_size)
out = lin2(tanlin)